# Accepted-loan preprocessing — V2

Steps 1–10 of `accepted_preprocessing.ipynb` are kept as they are. V2 replaces Steps 11–16.

| V1 problem | V2 fix |
|---|---|
| Labels never exported | `y_<split>.parquet` saved with `id`, `issue_d`, `target` |
| `earliest_cr_line` one-hot encoded (733 columns) | `credit_history_months` (1 numeric column) |
| Final encoders and medians never saved | one fitted `Pipeline` saved to `artifacts_v2/preprocessor_v2.joblib` |
| `zip_code` one-hot encoded (936 columns) | cross-fitted target encoding |
| Random split | temporal split by issue month |

**Splits:** train Aug-2012 → Dec-2014 · validation Jan → Jun 2015 · test Jul → Dec 2015 · 2016–2018 is a separate stress-test population only.

All logic lives in `src/preprocessing_v2/`. Each section below runs one stage. Outputs go only to `final_preprocessed_data_v2/` and `artifacts_v2/`. The V1 folders are never modified.

In [ ]:
import sys
from pathlib import Path

# Works whether the kernel starts in the repo root or in notebooks/
PROJECT_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "preprocessing_v2").is_dir())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd
from src.preprocessing_v2 import config, run, report

ctx = run.new_context()

## 1. Data restoration
Re-reads the raw CSV in its original order and applies the Step 3 target mapping. Before attaching `id`, `issue_d` and `target` by position, it checks that 12 shared columns are identical row by row with `accepted_quality_clean.csv`. Any mismatch stops the run.

In [ ]:
ctx = run.stage_restore(ctx)
ctx["restore_log"]

## 2. Temporal splitting
Rows issued before Aug-2012 are excluded, because the credit-bureau fields did not exist yet. 2016–2018 is exported only as a stress-test population: most of those loans had not finished by the 2018Q4 snapshot.

In [ ]:
ctx = run.stage_split(ctx)
ctx["split_summary"]

## 3. Feature engineering
Row-level only (nothing is fitted), so it is safe before the split. The same function runs inside the saved pipeline.
- `credit_history_months = (issue_year − ecl_year) × 12 + (issue_month − ecl_month)`
- `term_months`, `emp_length_years`
- `zip_code`, `emp_title`, `title` are stripped and lowercased for target encoding

In [ ]:
ctx = run.stage_feature_engineering(ctx)
ctx["engineering_preview"]

## 4. Fitted preprocessing (training data only)
`fit_transform(X_train, y_train)`: column filter → median imputation and missing flags → one-hot encoding → cross-fitted target encoding → duplicate-column removal. Validation and test are never passed to `fit`.

In [ ]:
ctx = run.stage_fit(ctx)
print(len(ctx["feature_names"]), "features")
pd.read_csv(config.SINGLE_FEATURE_AUC_PATH).head(10)

## 5. Artifact saving

In [ ]:
ctx = run.stage_save_artifacts(ctx)
ctx["preprocessor_path"]

## 6. Export
Each split is transformed, audited, written to Parquet, then read back and checked.

In [ ]:
ctx = run.stage_export(ctx)
ctx["dataset_paths"]

## 7. Final audits
Temporal ordering, a target spot check against the raw CSV, feature counts, the fresh-session inference test (a separate Python process), and a check that the backups are untouched. V2 is complete only if no required check fails. The single-feature AUC check is a review flag only.

In [ ]:
ctx = run.stage_final_audit(ctx)
report.write_reports(ctx)
print("PREPROCESSING V2:", "COMPLETE" if ctx["complete"] else "NOT COMPLETE")
ctx["audit"].frame()